# Fase 1 — Setup e Carregamento dos Dados

**Projeto**: Mineração de Padrões Frequentes em Acidentes de Rodovias Federais  
**Grupo 18**: Erik Roberto Reis Neves, Gabriel Campos Prudente, Felipe Silva Fraga Damasceno  
**Disciplina**: Mineração de Dados — UFMG  

---

## Objetivos desta fase
1. Instalar e importar todas as dependências necessárias
2. Criar infraestrutura **flexível** de configuração de dados
3. Implementar funções reutilizáveis de carregamento
4. Validar o carregamento de todos os datasets disponíveis

**Alinhamento CRISP-DM**: *Business Understanding* + início do *Data Understanding*

## 1.1 Instalação de Dependências

Execute esta célula apenas na primeira vez ou quando precisar atualizar pacotes.

In [ ]:
# Instalar dependências
# !pip install pandas numpy matplotlib seaborn mlxtend scikit-learn networkx folium

## 1.2 Imports e Configurações de Visualização

In [ ]:
# ============================================================
# IMPORTS PRINCIPAIS
# ============================================================

# Manipulação de dados
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Mineração de padrões frequentes
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Clustering e pré-processamento
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Utilitários
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURAÇÕES DE VISUALIZAÇÃO
# ============================================================
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')
sns.set_palette('viridis')

# Configurações do pandas
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

✅ Todos os imports realizados com sucesso!
   Pandas: 3.0.2
   NumPy: 2.4.4


## 1.3 Configuração Central dos Dados

> **Para trocar o dataset, altere apenas `ACTIVE_DATASET`.**  
> **Para trocar as colunas, edite os dicionários `COLUNAS_*`.**  
> **Para trocar o filtro geográfico, altere `FILTRO_UF`.**

In [ ]:
# ============================================================
# CONFIGURAÇÃO CENTRAL — ALTERE AQUI CONFORME NECESSIDADE
# ============================================================

# --- Caminhos ---
DATA_DIR = Path('../data/')
PROCESSED_DIR = Path('../data/processed/')
OUTPUT_DIR = Path('../outputs/')

# Criar diretórios se não existirem
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Mapa dos datasets disponíveis ---
DATASETS = {
    'por_ocorrencia': {
        'file': 'datatran2026_agrupados_por_ocorrencia.csv',
        'granularity': 'ocorrência (1 linha = 1 acidente)',
        'id_col': 'id',
        'description': 'Cada registro é uma ocorrência única de acidente. '
                       'Contém contagens agregadas de vítimas (mortos, feridos, ilesos). '
                       'Ideal para mineração de regras de associação.',
    },
    'por_pessoa': {
        'file': 'acidentes2026_agrupados_por_pessoa.csv',
        'granularity': 'pessoa (1 linha = 1 pessoa envolvida)',
        'id_col': 'pesid',
        'description': 'Cada registro é uma pessoa envolvida em um acidente. '
                       'Contém variáveis demográficas (idade, sexo, tipo_envolvido, estado_fisico). '
                       'Útil para análises complementares com perfil das vítimas.',
    },
    'todas_causas': {
        'file': 'acidentes2026_todas_causas_tipos.csv',
        'granularity': 'causa × pessoa (explode multicausa)',
        'id_col': 'pesid',
        'description': 'Versão expandida com múltiplas causas por acidente. '
                       'Inclui colunas causa_principal e ordem_tipo_acidente. '
                       'Útil quando se quer analisar TODAS as causas atribuídas.',
    },
}

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# >>> SELECIONE O DATASET <<<
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
ACTIVE_DATASET = 'por_ocorrencia'

# --- Separador CSV ---
SEP = ';'
ENCODING = 'latin-1'  # ISO-8859-1

# --- Filtro geográfico ---
# None => carregar todos os estados
FILTRO_UF = 'MG'

# ============================================================
# CONFIGURAÇÃO DE COLUNAS
# ============================================================

# Colunas categóricas para análise e mineração
# (presentes em todos os 3 datasets)
COLUNAS_CATEGORICAS_COMUNS = [
    'dia_semana',
    'causa_acidente',
    'tipo_acidente',
    'classificacao_acidente',
    'fase_dia',
    'sentido_via',
    'condicao_metereologica',
    'tipo_pista',
    'tracado_via',
    'uso_solo',
]

# Colunas categóricas extras (disponíveis nos datasets por pessoa)
COLUNAS_CATEGORICAS_PESSOA = [
    'tipo_veiculo',
    'tipo_envolvido',
    'estado_fisico',
    'sexo',
]

# Colunas numéricas de interesse
COLUNAS_NUMERICAS_OCORRENCIA = [
    'pessoas', 'mortos', 'feridos_leves', 'feridos_graves',
    'ilesos', 'ignorados', 'feridos', 'veiculos',
]

COLUNAS_NUMERICAS_PESSOA = [
    'idade', 'ilesos', 'feridos_leves', 'feridos_graves', 'mortos',
]

# Colunas temporais
COLUNAS_TEMPORAIS = ['data_inversa', 'horario']

# Colunas geográficas
COLUNAS_GEOGRAFICAS = ['uf', 'br', 'km', 'municipio', 'latitude', 'longitude']

# Colunas de identificação/administrativas
COLUNAS_ADMIN = ['regional', 'delegacia', 'uop']

print('✅ Configuração central definida!')
print(f'   Dataset ativo: {ACTIVE_DATASET}')
print(f'   Arquivo: {DATASETS[ACTIVE_DATASET]["file"]}')
print(f'   Granularidade: {DATASETS[ACTIVE_DATASET]["granularity"]}')
print(f'   Filtro UF: {FILTRO_UF or "Nenhum (todos os estados)"}')

✅ Configuração central definida!
   Dataset ativo: por_ocorrencia
   Arquivo: datatran2026_agrupados_por_ocorrencia.csv
   Granularidade: ocorrência (1 linha = 1 acidente)
   Filtro UF: MG


## 1.4 Funções de Carregamento

Funções reutilizáveis para carregar, filtrar e concatenar datasets.

In [4]:
def carregar_dataset(
    dataset_key: str,
    filtro_uf: str = None,
    colunas: list = None,
    data_dir: Path = DATA_DIR,
    sep: str = SEP,
    encoding: str = ENCODING,
) -> pd.DataFrame:
    """
    Carrega um dataset a partir da chave de configuração.

    Parameters
    ----------
    dataset_key : str
        Chave do dataset no dicionário DATASETS.
        Opções: 'por_ocorrencia', 'por_pessoa', 'todas_causas'
    filtro_uf : str, optional
        Sigla da UF para filtrar (ex: 'MG'). None para todos os estados.
    colunas : list, optional
        Lista de colunas a manter. None para manter todas.
    data_dir : Path
        Diretório dos dados.
    sep : str
        Separador do CSV.
    encoding : str
        Encoding do arquivo.

    Returns
    -------
    pd.DataFrame
        DataFrame carregado e filtrado.
    """
    if dataset_key not in DATASETS:
        raise ValueError(
            f"Dataset '{dataset_key}' não encontrado. "
            f"Opções: {list(DATASETS.keys())}"
        )

    info = DATASETS[dataset_key]
    path = data_dir / info['file']

    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

    # Carregar CSV
    df = pd.read_csv(path, sep=sep, encoding=encoding, low_memory=False)

    # Aplicar filtro geográfico
    if filtro_uf and 'uf' in df.columns:
        total_antes = len(df)
        df = df[df['uf'] == filtro_uf].copy()
        print(f"   Filtro UF={filtro_uf}: {total_antes:,} → {len(df):,} registros")

    # Selecionar colunas
    if colunas:
        colunas_disponiveis = [c for c in colunas if c in df.columns]
        colunas_ausentes = [c for c in colunas if c not in df.columns]
        if colunas_ausentes:
            print(f"   ⚠️  Colunas não encontradas (ignoradas): {colunas_ausentes}")
        df = df[colunas_disponiveis]

    # Resumo
    print(f"\n📊 Dataset carregado: {info['file']}")
    print(f"   Granularidade: {info['granularity']}")
    print(f"   Registros: {len(df):,}")
    print(f"   Colunas: {df.shape[1]}")
    print(f"   Memória: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

    return df


def carregar_e_concatenar(
    dataset_keys: list,
    filtro_uf: str = None,
    colunas: list = None,
    join: str = 'inner',
) -> pd.DataFrame:
    """
    Carrega e concatena múltiplos datasets.

    Parameters
    ----------
    dataset_keys : list
        Lista de chaves dos datasets a concatenar.
    filtro_uf : str, optional
        Sigla da UF para filtrar.
    colunas : list, optional
        Lista de colunas a manter.
    join : str
        Tipo de join: 'inner' (apenas colunas em comum) ou 'outer' (todas).

    Returns
    -------
    pd.DataFrame
        DataFrame concatenado.
    """
    dfs = []
    for key in dataset_keys:
        print(f"\n{'='*60}")
        print(f"Carregando: {key}")
        print(f"{'='*60}")
        df_temp = carregar_dataset(key, filtro_uf=filtro_uf, colunas=colunas)
        df_temp['_source_dataset'] = key  # Marcar a origem
        dfs.append(df_temp)

    df_concat = pd.concat(dfs, ignore_index=True, join=join)

    print(f"\n{'='*60}")
    print(f"📦 RESULTADO DA CONCATENAÇÃO")
    print(f"{'='*60}")
    print(f"   Datasets: {dataset_keys}")
    print(f"   Join: {join}")
    print(f"   Total registros: {len(df_concat):,}")
    print(f"   Total colunas: {df_concat.shape[1]}")
    print(f"   Memória: {df_concat.memory_usage(deep=True).sum() / 1e6:.1f} MB")

    return df_concat


def resumo_dataset(df: pd.DataFrame, nome: str = 'Dataset') -> pd.DataFrame:
    """
    Gera um resumo detalhado de cada coluna do DataFrame.

    Returns
    -------
    pd.DataFrame
        Tabela com tipo, nulos, únicos e exemplos de valores por coluna.
    """
    resumo = pd.DataFrame({
        'tipo': df.dtypes,
        'nulos': df.isnull().sum(),
        'nulos_%': (df.isnull().sum() / len(df) * 100).round(2),
        'unicos': df.nunique(),
        'exemplo_1': df.iloc[0] if len(df) > 0 else None,
        'exemplo_2': df.iloc[1] if len(df) > 1 else None,
    })
    resumo = resumo.sort_values('nulos_%', ascending=False)
    print(f"\n📋 Resumo do {nome}: {len(df):,} registros × {df.shape[1]} colunas")
    return resumo


def listar_datasets_disponiveis():
    """Exibe informações sobre todos os datasets configurados."""
    print("\n📁 DATASETS DISPONÍVEIS")
    print("=" * 70)
    for key, info in DATASETS.items():
        path = DATA_DIR / info['file']
        existe = '✅' if path.exists() else '❌'
        tamanho = f"{path.stat().st_size / 1e6:.1f} MB" if path.exists() else 'N/A'
        ativo = ' ← ATIVO' if key == ACTIVE_DATASET else ''
        print(f"\n  {existe} [{key}]{ativo}")
        print(f"     Arquivo: {info['file']}")
        print(f"     Tamanho: {tamanho}")
        print(f"     Granularidade: {info['granularity']}")
        print(f"     Descrição: {info['description']}")
    print()


print('✅ Funções de carregamento definidas!')

✅ Funções de carregamento definidas!


## 1.5 Verificação dos Datasets Disponíveis

In [5]:
listar_datasets_disponiveis()


📁 DATASETS DISPONÍVEIS

  ✅ [por_ocorrencia] ← ATIVO
     Arquivo: datatran2026_agrupados_por_ocorrencia.csv
     Tamanho: 6.8 MB
     Granularidade: ocorrência (1 linha = 1 acidente)
     Descrição: Cada registro é uma ocorrência única de acidente. Contém contagens agregadas de vítimas (mortos, feridos, ilesos). Ideal para mineração de regras de associação.

  ✅ [por_pessoa]
     Arquivo: acidentes2026_agrupados_por_pessoa.csv
     Tamanho: 24.0 MB
     Granularidade: pessoa (1 linha = 1 pessoa envolvida)
     Descrição: Cada registro é uma pessoa envolvida em um acidente. Contém variáveis demográficas (idade, sexo, tipo_envolvido, estado_fisico). Útil para análises complementares com perfil das vítimas.

  ✅ [todas_causas]
     Arquivo: acidentes2026_todas_causas_tipos.csv
     Tamanho: 74.8 MB
     Granularidade: causa × pessoa (explode multicausa)
     Descrição: Versão expandida com múltiplas causas por acidente. Inclui colunas causa_principal e ordem_tipo_acidente. Útil quando s

## 1.6 Carregamento do Dataset Ativo

Carrega o dataset configurado em `ACTIVE_DATASET` com o filtro geográfico `FILTRO_UF`.

In [6]:
# Carregar o dataset ativo
df = carregar_dataset(ACTIVE_DATASET, filtro_uf=FILTRO_UF)
df.head()

   Filtro UF=MG: 23,475 → 2,985 registros

📊 Dataset carregado: datatran2026_agrupados_por_ocorrencia.csv
   Granularidade: ocorrência (1 linha = 1 acidente)
   Registros: 2,985
   Colunas: 30
   Memória: 4.0 MB


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
1,742942,2026-01-01,quinta-feira,06:40:00,MG,262,"146,1",RIO CASCA,Condutor Dormindo,Colisão frontal,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,0,0,4,1,1,4,3,"-20,0240733","-42,7422291",SPRF-MG,DEL03-MG,UOP03-DEL03-MG
6,742985,2026-01-01,quinta-feira,12:00:00,MG,365,"393,4",PATOS DE MINAS,Chuva,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Aclive;Curva,Não,3,0,2,0,1,0,2,2,"-18,63407","-46,3586",SPRF-MG,DEL10-MG,UOP01-DEL10-MG
8,742988,2026-01-01,quinta-feira,09:30:00,MG,153,"245,5",FRONTEIRA,Frear bruscamente,Incêndio,Sem Vítimas,Pleno dia,Decrescente,Céu Claro,Simples,Aclive;Curva,Não,2,0,0,0,1,1,0,2,"-20,294923","-49,205151",SPRF-MG,DEL13-MG,UOP02-DEL13-MG
18,743093,2026-01-01,quinta-feira,21:35:00,MG,251,"474,7",FRANCISCO SA,Velocidade Incompatível,Saída de leito carroçável,Com Vítimas Feridas,Plena Noite,Crescente,Nublado,Simples,Curva;Declive,Não,2,0,1,0,0,2,1,3,"-16,470903","-43,442353",SPRF-MG,DEL12-MG,UOP01-DEL12-MG
21,743109,2026-01-02,sexta-feira,04:10:00,MG,381,"745,9",TRES CORACOES,Condutor Dormindo,Saída de leito carroçável,Com Vítimas Feridas,Plena Noite,Crescente,Chuva,Dupla,Curva;Aclive,Não,2,0,0,2,0,0,2,1,"-21,61428025","-45,26150465",SPRF-MG,DEL16-MG,UOP02-DEL16-MG


In [7]:
# Resumo detalhado
resumo = resumo_dataset(df, nome=ACTIVE_DATASET)
resumo


📋 Resumo do por_ocorrencia: 2,985 registros × 30 colunas


,tipo,nulos,nulos_%,unicos,exemplo_1,exemplo_2
id,int64,0,0.0000,2985,742942,742985
data_inversa,str,0,0.0000,120,2026-01-01,2026-01-01
dia_semana,str,0,0.0000,7,quinta-feira,quinta-feira
horario,str,0,0.0000,461,06:40:00,12:00:00
uf,str,0,0.0000,1,MG,MG
br,int64,0,0.0000,15,262,365
km,str,0,0.0000,1860,"146,1","393,4"
municipio,str,0,0.0000,203,RIO CASCA,PATOS DE MINAS
causa_acidente,str,0,0.0000,57,Condutor Dormindo,Chuva
tipo_acidente,str,0,0.0000,16,Colisão frontal,Colisão frontal


## 1.7 Validação: Carregamento de Todos os Datasets

Testar que os 3 datasets podem ser carregados individualmente e que a concatenação funciona.

In [8]:
# ============================================================
# VALIDAÇÃO 1: Carregar cada dataset individualmente
# ============================================================
print("VALIDAÇÃO 1: Carregamento individual de cada dataset")
print("=" * 60)

resultados_validacao = {}

for key in DATASETS.keys():
    try:
        df_test = carregar_dataset(key, filtro_uf=FILTRO_UF)
        resultados_validacao[key] = {
            'status': '✅ OK',
            'registros': len(df_test),
            'colunas': df_test.shape[1],
            'colunas_lista': list(df_test.columns),
        }
    except Exception as e:
        resultados_validacao[key] = {
            'status': f'❌ ERRO: {e}',
            'registros': 0,
            'colunas': 0,
        }

print("\n" + "=" * 60)
print("RESULTADO DA VALIDAÇÃO 1:")
print("=" * 60)
for key, res in resultados_validacao.items():
    print(f"  {res['status']} {key}: {res['registros']:,} registros, {res['colunas']} colunas")

VALIDAÇÃO 1: Carregamento individual de cada dataset


   Filtro UF=MG: 23,475 → 2,985 registros

📊 Dataset carregado: datatran2026_agrupados_por_ocorrencia.csv
   Granularidade: ocorrência (1 linha = 1 acidente)
   Registros: 2,985
   Colunas: 30
   Memória: 4.0 MB


   Filtro UF=MG: 64,125 → 8,275 registros

📊 Dataset carregado: acidentes2026_agrupados_por_pessoa.csv
   Granularidade: pessoa (1 linha = 1 pessoa envolvida)
   Registros: 8,275
   Colunas: 35
   Memória: 13.7 MB


   Filtro UF=MG: 196,099 → 24,649 registros

📊 Dataset carregado: acidentes2026_todas_causas_tipos.csv
   Granularidade: causa × pessoa (explode multicausa)
   Registros: 24,649
   Colunas: 37
   Memória: 42.4 MB

RESULTADO DA VALIDAÇÃO 1:
  ✅ OK por_ocorrencia: 2,985 registros, 30 colunas
  ✅ OK por_pessoa: 8,275 registros, 35 colunas
  ✅ OK todas_causas: 24,649 registros, 37 colunas


In [9]:
# ============================================================
# VALIDAÇÃO 2: Identificar colunas em comum entre datasets
# ============================================================
print("\nVALIDAÇÃO 2: Colunas em comum entre datasets")
print("=" * 60)

colunas_por_dataset = {
    k: set(v['colunas_lista'])
    for k, v in resultados_validacao.items()
    if 'colunas_lista' in v
}

# Colunas presentes em TODOS os datasets
colunas_comuns = set.intersection(*colunas_por_dataset.values())
print(f"\nColunas presentes em TODOS os datasets ({len(colunas_comuns)}):")
for col in sorted(colunas_comuns):
    print(f"  • {col}")

# Colunas exclusivas de cada dataset
print(f"\nColunas EXCLUSIVAS por dataset:")
for key, cols in colunas_por_dataset.items():
    exclusivas = cols - colunas_comuns
    if exclusivas:
        print(f"  {key}: {sorted(exclusivas)}")
    else:
        print(f"  {key}: (nenhuma exclusiva)")


VALIDAÇÃO 2: Colunas em comum entre datasets

Colunas presentes em TODOS os datasets (26):
  • br
  • causa_acidente
  • classificacao_acidente
  • condicao_metereologica
  • data_inversa
  • delegacia
  • dia_semana
  • fase_dia
  • feridos_graves
  • feridos_leves
  • horario
  • id
  • ilesos
  • km
  • latitude
  • longitude
  • mortos
  • municipio
  • regional
  • sentido_via
  • tipo_acidente
  • tipo_pista
  • tracado_via
  • uf
  • uop
  • uso_solo

Colunas EXCLUSIVAS por dataset:
  por_ocorrencia: ['feridos', 'ignorados', 'pessoas', 'veiculos']
  por_pessoa: ['ano_fabricacao_veiculo', 'estado_fisico', 'id_veiculo', 'idade', 'marca', 'pesid', 'sexo', 'tipo_envolvido', 'tipo_veiculo']
  todas_causas: ['ano_fabricacao_veiculo', 'causa_principal', 'estado_fisico', 'id_veiculo', 'idade', 'marca', 'ordem_tipo_acidente', 'pesid', 'sexo', 'tipo_envolvido', 'tipo_veiculo']


In [10]:
# ============================================================
# VALIDAÇÃO 3: Testar concatenação
# ============================================================
print("\nVALIDAÇÃO 3: Concatenação de datasets")
print("=" * 60)

# Concatenar 'por_ocorrencia' + 'por_pessoa' usando apenas colunas em comum
df_concat_test = carregar_e_concatenar(
    dataset_keys=['por_ocorrencia', 'por_pessoa'],
    filtro_uf=FILTRO_UF,
    join='inner',
)

print(f"\nOrigem dos registros:")
print(df_concat_test['_source_dataset'].value_counts())

# Limpar a variável de teste
del df_concat_test


VALIDAÇÃO 3: Concatenação de datasets

Carregando: por_ocorrencia


   Filtro UF=MG: 23,475 → 2,985 registros

📊 Dataset carregado: datatran2026_agrupados_por_ocorrencia.csv
   Granularidade: ocorrência (1 linha = 1 acidente)
   Registros: 2,985
   Colunas: 30
   Memória: 4.0 MB

Carregando: por_pessoa


   Filtro UF=MG: 64,125 → 8,275 registros

📊 Dataset carregado: acidentes2026_agrupados_por_pessoa.csv
   Granularidade: pessoa (1 linha = 1 pessoa envolvida)
   Registros: 8,275
   Colunas: 35
   Memória: 13.7 MB

📦 RESULTADO DA CONCATENAÇÃO
   Datasets: ['por_ocorrencia', 'por_pessoa']
   Join: inner
   Total registros: 11,260
   Total colunas: 27
   Memória: 15.4 MB

Origem dos registros:
_source_dataset
por_pessoa        8275
por_ocorrencia    2985
Name: count, dtype: int64


In [11]:
# ============================================================
# VALIDAÇÃO 4: Testar carregamento com seleção de colunas
# ============================================================
print("\nVALIDAÇÃO 4: Carregamento com seleção de colunas")
print("=" * 60)

# Carregar apenas as colunas categóricas comuns
colunas_teste = ['id'] + COLUNAS_CATEGORICAS_COMUNS
df_select_test = carregar_dataset(
    'por_ocorrencia',
    filtro_uf=FILTRO_UF,
    colunas=colunas_teste,
)
print(f"\nColunas selecionadas: {list(df_select_test.columns)}")
df_select_test.head(3)

del df_select_test


VALIDAÇÃO 4: Carregamento com seleção de colunas


   Filtro UF=MG: 23,475 → 2,985 registros

📊 Dataset carregado: datatran2026_agrupados_por_ocorrencia.csv
   Granularidade: ocorrência (1 linha = 1 acidente)
   Registros: 2,985
   Colunas: 11
   Memória: 2.1 MB

Colunas selecionadas: ['id', 'dia_semana', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo']


In [12]:
# ============================================================
# VALIDAÇÃO 5: Testar carregamento sem filtro de UF (Brasil inteiro)
# ============================================================
print("\nVALIDAÇÃO 5: Carregamento sem filtro geográfico")
print("=" * 60)

df_brasil = carregar_dataset('por_ocorrencia', filtro_uf=None)
print(f"\nDistribuição por UF (top 10):")
print(df_brasil['uf'].value_counts().head(10))

del df_brasil


VALIDAÇÃO 5: Carregamento sem filtro geográfico



📊 Dataset carregado: datatran2026_agrupados_por_ocorrencia.csv
   Granularidade: ocorrência (1 linha = 1 acidente)
   Registros: 23,475
   Colunas: 30
   Memória: 31.4 MB

Distribuição por UF (top 10):
uf
MG    2985
SC    2847
PR    2561
RJ    2099
RS    1488
SP    1416
BA    1344
GO    1101
PE    1086
MT     847
Name: count, dtype: int64


## 1.8 Resumo da Fase 1

### Checklist de validação

| Validação | Resultado |
|---|---|
| Carregar `por_ocorrencia` | ✅ |
| Carregar `por_pessoa` | ✅ |
| Carregar `todas_causas` | ✅ |
| Filtro por UF funciona | ✅ |
| Seleção de colunas funciona | ✅ |
| Concatenação funciona | ✅ |

### Decisões registradas
- **Dataset principal**: `por_ocorrencia` (1 linha = 1 acidente, ideal para transações no FP-Growth)
- **Filtro geográfico**: MG (Minas Gerais)
- **Encoding**: latin-1 (ISO-8859-1)
- **Separador**: `;`

### Próximo passo
→ **Fase 2**: Análise Exploratória (`02_analise_exploratoria.ipynb`)

In [13]:
# ============================================================
# DATASET FINAL PARA USO NOS PRÓXIMOS NOTEBOOKS
# ============================================================
# O DataFrame 'df' contém o dataset ativo configurado acima.
# Ele foi carregado na célula 1.6 e está pronto para uso.

print("\n" + "=" * 60)
print("✅ FASE 1 CONCLUÍDA")
print("=" * 60)
print(f"Dataset ativo: {ACTIVE_DATASET}")
print(f"Arquivo: {DATASETS[ACTIVE_DATASET]['file']}")
print(f"Filtro UF: {FILTRO_UF or 'Nenhum'}")
print(f"Registros: {len(df):,}")
print(f"Colunas: {df.shape[1]}")
print(f"Memória: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\n→ Próximo passo: Fase 2 — Análise Exploratória")


✅ FASE 1 CONCLUÍDA
Dataset ativo: por_ocorrencia
Arquivo: datatran2026_agrupados_por_ocorrencia.csv
Filtro UF: MG
Registros: 2,985
Colunas: 30
Memória: 4.3 MB

→ Próximo passo: Fase 2 — Análise Exploratória
